# EMIT column-wise reference SNR

Generate an experimental reference directly from native EMIT L1B RAD, or inspect
the distributed reference without satellite inputs. The selected scene is Libya-4,
2024-06-24 09:11:09 UTC, `20240624T091109_2417606_010`, V001 / build 010618.
See [the public catalog](../reference_snr/README.md) for all four sensors.

The CLI estimates a global four-component PCA reconstruction from all finite
SWIR spectra, subtracts it, and measures residual first differences along native
rows per band/column. SNR = mean observed radiance / (noise + 1e-12). No 12% mask,
destriping, QA exclusion or column filling is applied. All 2176 rows and 1242
columns are used by default. Generation may require about 12 GiB RAM.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/SNR_emit_reference.py').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from scripts.core.noise import ColumnwiseSNRReference

PUBLISHED = REPO / 'reference_snr/emit/20240624T091109_2417606_010/snr_reference_columnwise_experimental.npz'
RAD = REPO / 'downloads/emit/20240624T091109_2417606_010/EMIT_L1B_RAD_001_20240624T091109_2417606_010.nc'
OUTPUT = REPO / 'outputs/emit_reference_rebuilt'
RUN_GENERATION = False  # Enable after obtaining RAD; an existing OUTPUT is refused.


## Optional: regenerate from radiance

Change `RAD` to your product location. This calls the same public script as the
terminal command and compares numeric arrays with the distributed baseline.
For a different scene/configuration omit `--compare-reference` and validate it
separately. OBS and H2O are not required for reference SNR.


In [ ]:
if RUN_GENERATION:
    cmd = [sys.executable, str(REPO / 'scripts/SNR_emit_reference.py'),
           '--input', str(RAD), '--output-dir', str(OUTPUT),
           '--bands', '1000', '2500', '--k-pca', '4',
           '--compare-reference', str(PUBLISHED)]
    subprocess.run(cmd, check=True, cwd=REPO)
    print(json.dumps(json.loads((OUTPUT / 'generation_report.json').read_text()), indent=2))

reference_path = OUTPUT / PUBLISHED.name if RUN_GENERATION else PUBLISHED
reference = ColumnwiseSNRReference.load(reference_path)
print(reference_path, reference.snr.shape)
print('Native row range:', reference.metadata['native_row_range'])


## Inspect SNR and column support

Thin curves show each detector column; the black curve is the band-wise median
over columns with positive mean/SNR. The map preserves native band/column
positions. Unsupported or nonpositive cells stay unavailable for uncertainty.
These curves use the reference scene's own brightness, with no cross-sensor
brightness normalisation. Reference SNR characterises random noise, not every
striping component or total retrieval error.


In [ ]:
positive = (np.isfinite(reference.mean_radiance) & (reference.mean_radiance > 0)
            & np.isfinite(reference.snr) & (reference.snr > 0))
snr = np.where(positive, reference.snr, np.nan)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].plot(reference.band_nm, snr, color='C0', alpha=0.025, lw=0.5)
axes[0].plot(reference.band_nm, np.nanmedian(snr, axis=1), color='black', lw=1.5, label='Median across columns')
axes[0].set(xlabel='Wavelength (nm)', ylabel='Reference SNR', title='EMIT: native column curves')
axes[0].legend()
im = axes[1].imshow(snr, aspect='auto', origin='lower', interpolation='none')
axes[1].set(xlabel='Native detector column', ylabel='Reference band position', title='Positive radiometric support')
fig.colorbar(im, ax=axes[1], label='SNR')
plt.show()


In [ ]:
for gas, lo, hi in [('CH4 default', 2100, 2450), ('CO2 default', 1900, 2100), ('CO2 comparison', 1920, 2100)]:
    bands = (reference.band_nm >= lo) & (reference.band_nm <= hi)
    bad = ~positive[bands]
    print(f'{gas}: {bands.sum()} bands, {bad.sum()} unsupported cells, {bad.any(axis=0).sum()} affected columns')


## Use in retrieval

Pass `reference_path` through `--snr-reference` with EMIT RAD, paired OBS,
atmospheric L2A MASK and the CH4 or CO2 LUT. See the repository retrieval quick
start. The same radiometric reference serves both gases. This notebook and CLI
do not validate transfer to a different build/epoch or total plume uncertainty.
